# Evaluating the output of the SCIP solver of CVXPY

In [2]:
from av_mat_generation.CI_based.window import Window
from datetime import datetime
import pandas as pd
import numpy as np
import cvxpy as cp
import random

MAIN_FOLDER = 'availability_matrices/av-mat-scalability'
start_time=datetime(2022, 1, 1, 0, 0)
max_rounds=90
ft=1

In [3]:
# Diff nb clients: 
nb_clients_list = [7, 14, 28, 54]
# Diff base training time T (equiv to carbon budget)
T = [90, 75, 60, 45, 30, 15, 5, 3] #train time list
# Diff s: 
s_list = [20, 40, 60, 80, 100]

In [ ]:
for nb_clients in nb_clients_list:
    win = Window(start_time=start_time, out_folder=MAIN_FOLDER, n_rounds=max_rounds, custom_client_list=nb_clients)
    res = win.get_GHG_matrix()
    carbon_budget_considered = res.sum().cumsum().to_numpy().take([89,74,59,44,29,14,4,2])
    print(carbon_budget_considered)
    for ii, carbon_budget in enumerate(carbon_budget_considered):
        print(T[ii])
        for s in [20,40,60,80,100]:
            win1 = Window(start_time=start_time, out_folder=MAIN_FOLDER, n_rounds=T[ii]+s, custom_client_list=nb_clients)
            win = Window(start_time=start_time, out_folder=MAIN_FOLDER, n_rounds=T[ii]+s-ft, custom_client_list=nb_clients)
            try:
                av_mat = win.get_av_mat(method='cvxpy_scip', fine_tuning=False, carbon_budget=carbon_budget-win1.get_GHG_matrix().to_numpy()[:,-ft:].sum(), key_word=f"alphaF-{T[ii]+s}rnds-{ii+1}cb")
                availability_df = pd.DataFrame(np.hstack([av_mat.to_numpy(),np.ones((7,ft))]), index=win.countries, columns=[i for i in range(av_mat.shape[1]+ft)])
                key_word=f"alphaF-{T[ii]+s}rnds-{ii+1}cb-{ft}ft"
                dict_cols = dict(
                zip([i for i in range(win.n_rounds+ft)], win1.window_list_hours[:win.n_rounds+ft])
            )  # get the datetime values
                availability_matrix_to_save = availability_df.rename(columns=dict_cols)
                availability_matrix_to_save.to_csv(
                    win.out_folder + "/av-mat_" + key_word + ".csv",
                    columns=win1.window_list_hours[:win.n_rounds+ft],
                )
            except Exception as e:
                print(f"Failed! Exception: {e}")
                continue
        continue

[65.52236399999998 53.75549699999999 42.573671999999995 32.100399
 21.724482000000002 10.646117999999998 3.46971 2.091123]
90
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
75
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
60
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
Failed! Exception: The solver SCIP is not installed.
45
Failed! Exception: The solver SCIP is not installed.
Failed! Exception

In [4]:
import time

alpha_f = 0.1
timing_records = []

for nb_clients in nb_clients_list:
    win0 = Window(start_time=start_time, out_folder=MAIN_FOLDER, n_rounds=max_rounds, custom_client_list=nb_clients)
    res = win0.get_GHG_matrix()
    carbon_budget_considered = res.sum().cumsum().to_numpy().take([89,74,59,44,29,14,4,2])
    print(carbon_budget_considered)
    for ii, carbon_budget in enumerate(carbon_budget_considered):
        print(T[ii])
        for s in s_list:
            win1 = Window(start_time=start_time, out_folder=MAIN_FOLDER, n_rounds=T[ii]+s, custom_client_list=nb_clients)
            win = Window(start_time=start_time, out_folder=MAIN_FOLDER, n_rounds=T[ii]+s-ft, custom_client_list=nb_clients)
            corrected_budget = carbon_budget - win1.get_GHG_matrix().to_numpy()[:, -ft:].sum()
            record = dict(nb_clients=nb_clients, T=T[ii], s=s, n_rounds=win.n_rounds, corrected_budget=corrected_budget)
            try:
                # same problem construction as Window._av_mat_alphaF_cvxpy, without plotting/saving
                GHG_mat = win.get_GHG_matrix().to_numpy()
                w = np.ones(win.n_rounds)
                one_m_GHG_w = (np.max(GHG_mat) - GHG_mat) @ np.diag(w)

                t0 = time.perf_counter()
                x = cp.Variable(GHG_mat.shape, integer=True)
                objective = cp.Maximize(
                    cp.sum(cp.power(cp.sum(cp.multiply(one_m_GHG_w, x), axis=1), alpha_f))
                )
                constraints = [0 <= x, x <= 1, cp.sum(cp.multiply(GHG_mat, x)) <= corrected_budget]
                prob = cp.Problem(objective, constraints)
                t1 = time.perf_counter()
                prob.solve(solver=cp.SCIP, scip_params={"limits/totalnodes": 1000})
                t2 = time.perf_counter()

                record.update(
                    build_seconds=t1 - t0,
                    solve_seconds=t2 - t1,
                    scip_seconds=getattr(prob.solver_stats, "solve_time", None),
                    status=prob.status,
                    objective_value=prob.value,
                    achieved_GHG=float(np.sum(GHG_mat * x.value)) if x.value is not None else None,
                )
                print(f"{nb_clients} clients | T={T[ii]} s={s} ({win.n_rounds} rnds): solve {t2 - t1:.2f}s, status {prob.status}")
            except Exception as e:
                record.update(status=f"failed: {e}")
                print(f"Failed! Exception: {e}")
            timing_records.append(record)
            # keep partial results on disk in case the sweep is interrupted
            pd.DataFrame(timing_records).to_csv(MAIN_FOLDER + "/alphaF_solve_times.csv", index=False)

timing_df = pd.DataFrame(timing_records)
timing_df

[65.52236399999998 53.75549699999999 42.573671999999995 32.100399
 21.724482000000002 10.646117999999998 3.46971 2.091123]
90
7 clients | T=90 s=20 (109 rnds): solve 0.66s, status optimal


/home/danny/Documents/prjkts/GreenFL/venv/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


7 clients | T=90 s=40 (129 rnds): solve 0.94s, status optimal_inaccurate


/home/danny/Documents/prjkts/GreenFL/venv/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


7 clients | T=90 s=60 (149 rnds): solve 1.71s, status optimal_inaccurate


...

54 clients | T=3 s=60 (62 rnds): solve 18.95s, status optimal_inaccurate


/home/danny/Documents/prjkts/GreenFL/venv/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


54 clients | T=3 s=80 (82 rnds): solve 23.60s, status optimal_inaccurate
54 clients | T=3 s=100 (102 rnds): solve 18.26s, status optimal_inaccurate


/home/danny/Documents/prjkts/GreenFL/venv/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


,nb_clients,T,s,n_rounds,corrected_budget,build_seconds,solve_seconds,scip_seconds,status,objective_value,achieved_GHG
0,7,90,20,109,64.760184,0.000628,0.663568,None,optimal,9.206293,64.726476
1,7,90,40,129,64.765383,0.000538,0.942298,None,optimal_inaccurate,9.263166,64.764321
2,7,90,60,149,64.746738,0.000476,1.714391,None,optimal_inaccurate,9.296290,64.742961
3,7,90,80,169,64.722960,0.000576,3.929349,None,optimal_inaccurate,9.323338,64.719288
4,7,90,100,189,64.737717,0.000459,1.968048,None,optimal_inaccurate,9.349387,64.735017
...,...,...,...,...,...,...,...,...,...,...,...
155,54,3,20,22,9.381372,0.000638,10.339164,None,optimal_inaccurate,51.994159,9.381372
156,54,3,40,42,9.382149,0.000735,12.354662,None,optimal_inaccurate,52.579863,9.381924
157,54,3,60,62,8.976972,0.000576,18.948081,None,optimal_inaccurate,52.542582,8.976303
158,54,3,80,82,8.938476,0.000606,23.604551,None,optimal_inaccurate,52.638353,8.935941
